# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Devaaldo/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!pip install -q duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get("HF_TOKEN")
con.sql(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{hf_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{BASE}/fact_content_daily_performance/train/*.parquet"
CLIENTS = f"{BASE}/dim_clients.parquet"

MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03"]
FACT_FILES = [f"{BASE}/fact_content_daily_performance/month={m}/data_0.parquet" for m in MONTHS]

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Window: {WINDOW_START} to {WINDOW_END} (90 days, trailing, ending mid-panel)")

Window: 2025-12-31 to 2026-03-31 (90 days, trailing, ending mid-panel)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fields = {
    "feature": ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "gsc_data_available"],
    "label":   ["clicks_last30", "clicks_prev30 (derived → is_declining_label)"],
    "context": ["client_hash_id", "content_hash_id", "report_date"],
    "excluded":["ga4_* columns — out of scope for search-signal lane; not knowable pre-GA4-linkage for all clients"],
}
fields

{'feature': ['gsc_impressions',
  'gsc_clicks',
  'gsc_avg_position',
  'gsc_data_available'],
 'label': ['clicks_last30', 'clicks_prev30 (derived → is_declining_label)'],
 'context': ['client_hash_id', 'content_hash_id', 'report_date'],
 'excluded': ['ga4_* columns — out of scope for search-signal lane; not knowable pre-GA4-linkage for all clients']}

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT COUNT(*) AS n_daily_rows,
       COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS n_unique_grain
FROM read_parquet({FACT_FILES})
WHERE report_date BETWEEN DATE '{WINDOW_START}' AND DATE '{WINDOW_END}'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬────────────────┐
│ n_daily_rows │ n_unique_grain │
│    int64     │     int64      │
├──────────────┼────────────────┤
│     25338946 │       25338946 │
└──────────────┴────────────────┘



In [17]:
con.sql(f"""
SELECT COUNT(*) AS n_rows_aggregated
FROM (
    SELECT client_hash_id, content_hash_id
    FROM read_parquet({FACT_FILES})
    WHERE report_date BETWEEN DATE '{WINDOW_START}' AND DATE '{WINDOW_END}'
    GROUP BY 1, 2
) sub
""").show()

con.sql(f"""
SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest
FROM read_parquet({FACT_FILES})
WHERE report_date BETWEEN DATE '{WINDOW_START}' AND DATE '{WINDOW_END}'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┐
│ n_rows_aggregated │
│       int64       │
├───────────────────┤
│            349411 │
└───────────────────┘

┌────────────┬────────────┐
│  earliest  │   latest   │
│    date    │    date    │
├────────────┼────────────┤
│ 2025-12-31 │ 2026-03-31 │
└────────────┴────────────┘



In [14]:
con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS survives
FROM read_parquet({FACT_FILES})
WHERE report_date BETWEEN DATE '{WINDOW_START}' AND DATE '{WINDOW_END}'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────┐
│ total_rows │ survives │
│   int64    │  int128  │
├────────────┼──────────┤
│   25338946 │  8700314 │
└────────────┴──────────┘



In [15]:
df = con.sql(f"""
SELECT
  client_hash_id, content_hash_id,
  SUM(gsc_impressions) AS impressions_90d,
  SUM(gsc_clicks)      AS clicks_90d,
  AVG(gsc_avg_position) AS avg_position_90d,
  SUM(CASE WHEN report_date > DATE '2026-02-28' THEN gsc_clicks ELSE 0 END) AS clicks_last30,
  SUM(CASE WHEN report_date <= DATE '2026-02-28' THEN gsc_clicks ELSE 0 END) AS clicks_prev30,
  COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_90d
FROM read_parquet({FACT_FILES})
WHERE report_date BETWEEN DATE '{WINDOW_START}' AND DATE '{WINDOW_END}'
  AND gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

df["is_declining_label"] = (df["clicks_last30"] < df["clicks_prev30"]).astype(int)

import pandas as pd
feature_cols = ["impressions_90d", "clicks_90d", "avg_position_90d", "active_days_90d"]
df["ctr_90d"] = df["clicks_90d"] / df["impressions_90d"].replace(0, pd.NA)
feature_cols.append("ctr_90d")

df[feature_cols + ["is_declining_label"]].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impressions_90d,clicks_90d,avg_position_90d,active_days_90d,ctr_90d,is_declining_label
0,3228.0,4.0,17.644703,89,0.001239,0
1,8828.0,37.0,11.289451,89,0.004191,1
2,2088.0,0.0,38.396129,89,0.000000,0
3,28718.0,15.0,41.073000,89,0.000522,1
4,4704.0,7.0,19.257231,89,0.001488,0
5,3307.0,3.0,53.240000,89,0.000907,0
6,19092.0,13.0,9.233377,89,0.000681,1
7,282.0,1.0,14.979798,74,0.003546,1
8,2791.0,1.0,37.395915,83,0.000358,1
9,2730.0,0.0,50.352775,89,0.000000,0


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X_honest = df[feature_cols].fillna(0)
y = df["is_declining_label"]

score_honest = cross_val_score(LogisticRegression(max_iter=1000), X_honest, y, cv=5).mean()
print("Honest score:", score_honest)

X_leaky = X_honest.copy()
X_leaky["leak_clicks_last30"] = df["clicks_last30"]  # ini bagian pembentuk label!

score_leaky = cross_val_score(LogisticRegression(max_iter=1000), X_leaky, y, cv=5).mean()
print("Leaky score:", score_leaky)  # harus jauh lebih tinggi

Honest score: 0.810009890739974
Leaky score: 1.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT client_hash_id, MIN(report_date) AS first_seen, MAX(report_date) AS last_seen,
       DATEDIFF('day', MIN(report_date), MAX(report_date)) AS days_span
FROM read_parquet({FACT_FILES})
GROUP BY 1
ORDER BY days_span
LIMIT 10
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬────────────┬────────────┬───────────┐
│     client_hash_id      │ first_seen │ last_seen  │ days_span │
│         varchar         │    date    │    date    │   int64   │
├─────────────────────────┼────────────┼────────────┼───────────┤
│ client_e00b29e582949543 │ 2026-03-23 │ 2026-03-31 │         8 │
│ client_810019792c9b8efc │ 2026-03-20 │ 2026-03-31 │        11 │
│ client_f6f0cdf26d03d7bd │ 2026-03-19 │ 2026-03-31 │        12 │
│ client_86ebc2f12c01f586 │ 2026-03-03 │ 2026-03-31 │        28 │
│ client_b77d0d5f08f05e64 │ 2026-03-01 │ 2026-03-31 │        30 │
│ client_7eafe750768f0ac2 │ 2026-02-26 │ 2026-03-31 │        33 │
│ client_a80fca3f171ed1de │ 2026-02-19 │ 2026-03-31 │        40 │
│ client_157ffe4d4a595515 │ 2026-02-19 │ 2026-03-31 │        40 │
│ client_e5c2aa26a8598242 │ 2026-02-19 │ 2026-03-31 │        40 │
│ client_20259bd6705d81d4 │ 2026-02-19 │ 2026-03-31 │        40 │
├─────────────────────────┴────────────┴────────────┴───────────┤
│ 10 rows 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.